# FarmTech Solutions — Fase 6: Visão Computacional
## Ir Além 2 — Transfer Learning × CNN do zero, com e sem segmentação

**Grupo:** Grupo 3
**Integrantes:** Gerson Ferreira da Graça (RM 569624), Lucas Braga (RM 568712), Carlos Leonardo Mazieri (RM 572809), Ryann Pinto (RM 571003)
**Objetos:** Tomate (classe 0) e Pimentão (classe 1)

> **Situação deste notebook:** código pronto para as **4 combinações** — ambiente, dataset e segmentação (seções 1 a 3), CNN do zero com segmentação (seção 4) e Transfer Learning com Fine Tuning sem e com segmentação (seções 5 e 6). Faltam a execução dos treinos com GPU, a tabela final (seção 7) e as conclusões (seção 8).

---

### 1. Objetivo e contexto

A Entrega 2 treinou uma **CNN do zero** para classificar a imagem inteira (tomate ou pimentão). Este notebook testa duas hipóteses sobre a mesma base de 80 imagens e a **mesma divisão** das Entregas 1 e 2 (32 treino / 4 validação / 4 teste por classe):

1. **Uma rede pré-treinada classifica melhor que uma treinada do zero?** — *Transfer Learning* com **Fine Tuning** da **MobileNetV2** pré-treinada na ImageNet, em duas fases: primeiro só o classificador novo é treinado, com a rede base congelada; depois as **últimas 20 camadas** da base (blocos 15 e 16 e a convolução final) são descongeladas e ajustadas com uma taxa de aprendizado 10 vezes menor.
2. **Separar o objeto do fundo antes de classificar ajuda?** — **segmentação com uma rede neural, a U2-Net** (via biblioteca `rembg`): ela cria a máscara do objeto principal da foto sem nenhuma rotulagem manual nova; a máscara apaga o fundo e recorta a imagem no objeto antes de ela entrar na rede de classificação.

O que o enunciado pede, nas palavras dele:

> *"Aplicar uma rede de segmentação do objeto desejado, criando uma máscara; Usar a máscara obtida para 'cortar' a imagem original, deixando apenas o objeto principal e apagando o background [...] Realizar Fine Tuning, justificar a escolha da camada de congelamento."* — enunciado do Ir Além 2

As duas hipóteses se cruzam em 4 combinações, todas avaliadas nas mesmas 8 imagens de teste:

| | **Sem segmentação** | **Com segmentação (U2-Net)** |
|:--|:--:|:--:|
| **CNN do zero** | resultado da Entrega 2 (não é treinada de novo) | seção 4 |
| **Transfer Learning + Fine Tuning (MobileNetV2)** | seção 5 | seção 6 |

**Por que MobileNetV2:** é leve (cerca de 2,26 milhões de parâmetros na parte convolucional), tem pesos da ImageNet prontos no Keras e aceita entrada de 128×128 — o mesmo tamanho usado pela CNN da Entrega 2, o que mantém a comparação justa.

**Por que U2-Net:** é uma rede neural de segmentação de objeto saliente, já pré-treinada para separar o objeto principal de uma foto do fundo — exatamente o caso do nosso dataset (um fruto por foto, de perto). Funciona sem prompt e sem máscaras desenhadas à mão: o dataset tem caixas (YOLO), mas não máscaras.

## 1.1 Setup do ambiente

Use um ambiente com **GPU T4** (Ambiente de execução > Alterar tipo de ambiente de execução). O TensorFlow já vem instalado no Colab; a célula seguinte instala só a biblioteca `rembg`, que traz a U2-Net.

In [ ]:
# Conectar ao Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Caminho base do projeto no Drive (o mesmo das Entregas 1 e 2).
# Quem NÃO é o dono da pasta precisa, uma única vez, criar o atalho:
# "Compartilhados comigo" → botão direito em FarmTech_Fase6 → "Organizar" → "Adicionar atalho" → "Meu Drive".
BASE_PATH = '/content/drive/MyDrive/FarmTech_Fase6'

# Identificação de quem está rodando o notebook.
# Cada integrante troca para o próprio primeiro nome (minúsculo, sem acento) ANTES de rodar.
# Tudo o que este notebook grava vai para Resultados/ir_alem_2_<EXECUTOR>/.
EXECUTOR = 'gerson'
PASTA_IA2 = f'{BASE_PATH}/Resultados/ir_alem_2_{EXECUTOR}'

In [ ]:
# Biblioteca da rede de segmentação (U2-Net). A versão é fixa para a execução ser reproduzível.
# Se o Colab mostrar o aviso de reiniciar a sessão depois da instalação, clique em "Reiniciar sessão" e execute tudo de novo.
!pip install -q "rembg[cpu]==2.0.85"

import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image, ImageOps

gpus = tf.config.list_physical_devices('GPU')
print('GPU disponível:', bool(gpus), gpus)
if not gpus:
    print('ATENÇÃO: sem GPU — troque para GPU T4 em Ambiente de execução > Alterar tipo de ambiente de execução.')
print('tensorflow', tf.__version__)

os.makedirs(PASTA_IA2, exist_ok=True)
print('Saídas deste notebook em:', PASTA_IA2)

## 2. Carregamento do dataset

As fotos são lidas direto das pastas por classe do Drive (`Dataset_dividido/{Tomate,Pimentao}/{Treino,Validacao,Teste}`) — a **mesma divisão** usada pela CNN da Entrega 2. Nenhuma imagem é copiada ou alterada no Drive.

In [ ]:
# 2.0 Conferir a divisão do dataset (falha cedo se algo estiver errado)
ORIGEM = os.path.join(BASE_PATH, 'Dataset_dividido')
CLASSES = ['tomate', 'pimentao']                       # índice 0 e 1 — mesma ordem das Entregas 1 e 2
PASTA_CLASSE = {'tomate': 'Tomate', 'pimentao': 'Pimentao'}
SPLITS = {'treino': 'Treino', 'validacao': 'Validacao', 'teste': 'Teste'}
ESPERADO = {'treino': 32, 'validacao': 4, 'teste': 4}  # por classe
EXTENSOES = ('.jpg', '.jpeg', '.png')


def nome_normalizado(arquivo):
    # 'Tomate 16.jpg' → 'tomate_16' (mesma regra das Entregas 1 e 2)
    return '_'.join(os.path.splitext(arquivo)[0].lower().split())


# Lista de (caminho, índice da classe, nome normalizado) por split
amostras = {split: [] for split in SPLITS}
problemas = []
for split, pasta_split in SPLITS.items():
    for indice, classe in enumerate(CLASSES):
        pasta = os.path.join(ORIGEM, PASTA_CLASSE[classe], pasta_split)
        arquivos = sorted(f for f in os.listdir(pasta) if f.lower().endswith(EXTENSOES))
        if len(arquivos) != ESPERADO[split]:
            problemas.append(f'{PASTA_CLASSE[classe]}/{pasta_split}: {len(arquivos)} imagens (esperado {ESPERADO[split]})')
        amostras[split] += [(os.path.join(pasta, f), indice, nome_normalizado(f)) for f in arquivos]
    print(f'{split:<9}: {len(amostras[split]):>2} imagens')

if problemas:
    raise RuntimeError('Problemas no dataset:\n- ' + '\n- '.join(problemas))

TESTE = amostras['teste']   # ordem fixa: 4 tomates, depois 4 pimentões — a mesma da Entrega 2
print('\n✅ Dataset conferido. Imagens de teste:', ', '.join(nome for _, _, nome in TESTE))

In [ ]:
# 2.1 Carregar as fotos originais em memória (a segmentação trabalha sobre a foto inteira)
TAMANHO = 128   # entrada das redes — o mesmo tamanho da CNN da Entrega 2


def carregar_original(caminho):
    """Foto em RGB (uint8), já na orientação correta (exif_transpose aplica a rotação gravada pelo celular)."""
    return np.asarray(ImageOps.exif_transpose(Image.open(caminho)).convert('RGB'))


def redimensionar(imagem):
    """Imagem RGB (qualquer tamanho) → array float32 TAMANHO×TAMANHO, pixels de 0 a 255."""
    return np.asarray(Image.fromarray(imagem).resize((TAMANHO, TAMANHO)), dtype=np.float32)


originais = {split: [carregar_original(c) for c, _, _ in lista] for split, lista in amostras.items()}
rotulos = {split: np.array([i for _, i, _ in lista], dtype=np.float32) for split, lista in amostras.items()}

# Versão SEM segmentação: a foto inteira redimensionada (exatamente o que a CNN da Entrega 2 recebeu)
X_sem = {split: np.stack([redimensionar(img) for img in imgs]) for split, imgs in originais.items()}
for split in SPLITS:
    print(f'{split:<9}: X {X_sem[split].shape} | pimentões: {int(rotulos[split].sum())}')

## 3. Segmentação automática com a U2-Net

> *"Aplicar uma rede de segmentação do objeto desejado, criando uma máscara; Usar a máscara obtida para 'cortar' a imagem original, deixando apenas o objeto principal e apagando o background [...] Realizar Fine Tuning, justificar a escolha da camada de congelamento."* — enunciado do Ir Além 2

**Como funciona:** a **U2-Net** é uma rede neural de segmentação de objeto saliente — ela estima, para cada pixel da foto, a chance de ele pertencer ao objeto principal. A biblioteca `rembg` traz essa rede já treinada (o modelo, de cerca de 176 MB, é baixado na primeira execução). Não há prompt, retângulo inicial nem rotulagem manual: a rede decide sozinha o que é objeto.

**Etapas da função `segmentar_u2net`:**
1. a rede devolve uma máscara de probabilidade (0 a 255) do tamanho da foto;
2. os pixels com probabilidade de pelo menos 50% formam a máscara binária do objeto;
3. o fundo é pintado de preto e a imagem é recortada no retângulo que envolve o objeto;
4. se a máscara cobrir menos de 5% da foto, a segmentação é considerada falha e a foto original é devolvida.

**Por que não GrabCut:** a primeira versão deste notebook usava GrabCut (OpenCV), um algoritmo clássico — não uma rede neural, como o enunciado pede. Além disso, num teste com as 8 imagens de teste, executado fora do Colab, o GrabCut errou em 3 fotos com fundo carregado ou fruto fora do centro, e a U2-Net acertou o objeto nas 8:

| Imagem | GrabCut | U2-Net |
|:--|:--|:--|
| tomate_23 | levou junto parte do piso | só o tomate |
| tomate_29 | pegou **a parede** no lugar do tomate | o tomate |
| pimentao_12 | levou junto a parede ao lado | só o pimentão |
| pimentao_36 | cortou o topo do pimentão | o pimentão inteiro |
| demais 4 imagens | objeto certo | objeto certo (máscaras quase iguais às do GrabCut) |

O custo medido nesse teste foi de cerca de 0,47 s por imagem em CPU.

In [ ]:
# 3.0 Função de segmentação com a U2-Net
from rembg import new_session, remove

SESSAO_U2NET = new_session('u2net')   # baixa o modelo pré-treinado na 1ª vez (≈176 MB) e reaproveita nas demais


def segmentar_u2net(imagem, limiar=0.5, area_minima=0.05):
    """Separa o objeto do fundo com a U2-Net (rede de segmentação de objeto saliente), sem rótulo nem prompt manual.

    Devolve (recortada, mascara, ok):
      recortada — imagem só com o objeto (fundo preto), cortada no retângulo que envolve a máscara
      mascara   — máscara binária (uint8, 0/1) no tamanho da imagem original
      ok        — False se a máscara ficou pequena demais (< 5% da foto); nesse caso a recortada é a original
    """
    probabilidade = np.asarray(remove(imagem, session=SESSAO_U2NET, only_mask=True))
    mascara = (probabilidade >= limiar * 255).astype(np.uint8)
    if mascara.mean() < area_minima:
        return imagem, mascara, False
    ys, xs = np.where(mascara)
    so_objeto = imagem * mascara[:, :, None]
    return so_objeto[ys.min():ys.max() + 1, xs.min():xs.max() + 1], mascara, True

In [ ]:
# 3.1 Teste visual: original | máscara | recortada — as mesmas 3 imagens do teste com o GrabCut
# (2 casos típicos e o tomate_29, fruto fora do centro, em que o GrabCut pegou a parede)
EXEMPLOS = ['tomate_17', 'pimentao_36', 'tomate_29']
por_nome = {nome: (split, k) for split, lista in amostras.items() for k, (_, _, nome) in enumerate(lista)}

fig, eixos = plt.subplots(len(EXEMPLOS), 3, figsize=(10, 3.4 * len(EXEMPLOS)))
for linha, nome in enumerate(EXEMPLOS):
    split, k = por_nome[nome]
    recortada, mascara, ok = segmentar_u2net(originais[split][k])
    for coluna, (imagem, titulo) in enumerate([(originais[split][k], f'{nome} — original'),
                                               (mascara, f'máscara ({mascara.mean():.0%} da foto)'),
                                               (recortada, 'recortada' + ('' if ok else ' (falhou: original)'))]):
        eixos[linha, coluna].imshow(imagem, cmap='gray' if imagem.ndim == 2 else None)
        eixos[linha, coluna].set_title(titulo, fontsize=10)
        eixos[linha, coluna].axis('off')
plt.tight_layout()
plt.savefig(f'{PASTA_IA2}/u2net_exemplos.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 3.2 Versão COM segmentação de todas as imagens (calculada uma vez e reaproveitada nas seções 4 e 6)
inicio = time.perf_counter()
segmentacao = {split: [segmentar_u2net(img) for img in imgs] for split, imgs in originais.items()}
TEMPO_SEGMENTACAO_TOTAL_S = time.perf_counter() - inicio

X_com = {split: np.stack([redimensionar(recortada) for recortada, _, _ in res]) for split, res in segmentacao.items()}
falhas = {split: [amostras[split][k][2] for k, (_, _, ok) in enumerate(res) if not ok] for split, res in segmentacao.items()}
n_imagens = sum(len(v) for v in originais.values())
print(f'U2-Net em {n_imagens} imagens: {TEMPO_SEGMENTACAO_TOTAL_S:.1f} s (≈{TEMPO_SEGMENTACAO_TOTAL_S / n_imagens * 1000:.0f} ms por imagem)')
print('Segmentações devolvidas como "falha" (máscara < 5%):', {s: f for s, f in falhas.items() if f} or 'nenhuma')

## 4. CNN do zero **com** segmentação

**O que está sendo testado:** a **mesma CNN do zero da Entrega 2** — mesmas camadas, mesmos hiperparâmetros e mesmo critério de parada —, agora recebendo as imagens **pré-segmentadas pela U2-Net** (`X_com`: fundo apagado e foto recortada no fruto). A única diferença em relação à Entrega 2 é a entrada.

- entrada 128×128×3; aumento de dados (flip, rotação 0,15, zoom 0,15); `Rescaling(1/255)`;
- 3 blocos `Conv2D` (16, 32, 64 filtros) + `MaxPooling2D`; `GlobalAveragePooling2D`; `Dropout(0.3)`; `Dense(1, sigmoid)`;
- Adam (1e-3), `binary_crossentropy`, até 100 épocas, `batch_size=8`, semente 0;
- parada antecipada por `val_accuracy` (paciência 30, restaurando os melhores pesos) — o mesmo critério corrigido da Entrega 2.

**Hipótese:** com só 64 fotos de treino, uma rede que aprende do zero pode se apoiar no **fundo** — piso, parede, mesa — para separar as classes, em vez de olhar o fruto. Apagando o fundo, sobra só o objeto: a expectativa é que a CNN se concentre em cor, formato e textura do tomate ou do pimentão. O efeito contrário também é possível — o recorte muda a escala do fruto na imagem e tira o contexto —, e é isso que a comparação vai medir.

A combinação **CNN do zero sem segmentação** não é treinada de novo: usa o resultado registrado na Entrega 2 (7 de 8 no teste).

In [ ]:
# 4.0 Imports do Keras e função de avaliação no teste (usada nas seções 4, 5 e 6)
from tensorflow import keras
from tensorflow.keras import layers

LIMIAR = 0.5   # probabilidade de pimentão a partir da qual a resposta é "pimentão" (o mesmo da Entrega 2)


def avaliar_teste(modelo, X_teste, preparo, rodadas=5):
    """Resposta da rede em cada imagem de teste + tempo de inferência por imagem (mediana, uma imagem por vez).

    preparo: função que leva a foto original até a entrada da rede — só redimensionar, ou segmentar e redimensionar.
    Assim, nas versões com segmentação, o tempo da U2-Net entra na conta.
    """
    probabilidade = modelo.predict(X_teste, verbose=0).ravel()
    tabela = pd.DataFrame({
        'imagem': [nome for _, _, nome in TESTE],
        'classe real': [CLASSES[i] for _, i, _ in TESTE],
        'P(pimentão)': probabilidade.round(3),
        'resposta': [CLASSES[int(p >= LIMIAR)] for p in probabilidade],
    })
    tabela['acertou?'] = tabela['resposta'] == tabela['classe real']

    def responder(imagem):
        return float(modelo(preparo(imagem)[None], training=False)[0, 0])

    for imagem in originais['teste']:   # aquecimento
        responder(imagem)
    tempos = []
    for _ in range(rodadas):
        for imagem in originais['teste']:
            inicio = time.perf_counter()
            responder(imagem)
            tempos.append((time.perf_counter() - inicio) * 1000)
    return tabela, float(np.median(tempos))

In [ ]:
# 4.1 CNN do zero — exatamente a arquitetura e o treino da Entrega 2 (células 6.1 e 6.2); só a entrada muda: X_com
def montar_cnn():
    keras.utils.set_random_seed(0)
    return keras.Sequential([
        keras.Input(shape=(TAMANHO, TAMANHO, 3)),
        # Aumento de dados — ativo só durante o treino
        layers.RandomFlip('horizontal_and_vertical'),
        layers.RandomRotation(0.15),
        layers.RandomZoom(0.15),
        # Pixels de 0–255 para 0–1
        layers.Rescaling(1. / 255),
        # Bloco 1: bordas e cores
        layers.Conv2D(16, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(),
        # Bloco 2: texturas e partes do contorno
        layers.Conv2D(32, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(),
        # Bloco 3: formas mais completas
        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(),
        # Classificador
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid'),   # probabilidade de ser pimentão
    ], name='cnn_do_zero')


EPOCAS_MAX = 100   # mesmos critérios da Entrega 2
PACIENCIA = 30     # parada por acurácia de validação — com só 8 imagens, a val_loss não serve para escolher a época

cnn_com = montar_cnn()
cnn_com.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
parada = keras.callbacks.EarlyStopping(monitor='val_accuracy', mode='max', patience=PACIENCIA, restore_best_weights=True)

inicio = time.perf_counter()
historico_cnn_com = cnn_com.fit(X_com['treino'], rotulos['treino'], validation_data=(X_com['validacao'], rotulos['validacao']),
                                epochs=EPOCAS_MAX, batch_size=8, callbacks=[parada], verbose=2)
TEMPO_TREINO_CNN_COM_S = time.perf_counter() - inicio

# np.argmax devolve a PRIMEIRA época com a maior val_accuracy — a mesma que o Keras restaura
melhor_epoca_cnn_com = int(np.argmax(historico_cnn_com.history['val_accuracy'])) + 1
print(f"\nTreino: {TEMPO_TREINO_CNN_COM_S:.1f} s | épocas rodadas: {len(historico_cnn_com.history['loss'])} | "
      f"melhor época (maior val_accuracy): {melhor_epoca_cnn_com}")

# Guardar no Drive (para recarregar sem treinar de novo: keras.models.load_model(...))
cnn_com.save(f'{PASTA_IA2}/cnn_do_zero_com_segmentacao.keras')
pd.DataFrame(historico_cnn_com.history).to_csv(f'{PASTA_IA2}/cnn_do_zero_com_segmentacao_historico.csv', index_label='epoca')

In [ ]:
# 4.2 Resultado no teste — CNN do zero com segmentação (o tempo inclui a U2-Net)
tabela_cnn_com, inferencia_cnn_com_ms = avaliar_teste(
    cnn_com, X_com['teste'], preparo=lambda imagem: redimensionar(segmentar_u2net(imagem)[0]))
display(tabela_cnn_com)
print(f"Acurácia no teste: {tabela_cnn_com['acertou?'].sum()}/8 | inferência: {inferencia_cnn_com_ms:.1f} ms por imagem (mediana, com a U2-Net)")

## 5. Transfer Learning + Fine Tuning com MobileNetV2 — **sem** segmentação

> *"Aplicar uma rede de segmentação do objeto desejado, criando uma máscara; Usar a máscara obtida para 'cortar' a imagem original, deixando apenas o objeto principal e apagando o background [...] Realizar Fine Tuning, justificar a escolha da camada de congelamento."* — enunciado do Ir Além 2

**Modelo:** `MobileNetV2(input_shape=(128, 128, 3), include_top=False, weights='imagenet')` + o mesmo aumento de dados da CNN (flip, rotação 0,15, zoom 0,15) + conversão dos pixels de 0–255 para −1 a 1 (o formato em que a rede foi treinada na ImageNet) + classificador novo: `GlobalAveragePooling2D` → `Dropout(0.3)` → `Dense(1, sigmoide)`.

**Treino em duas fases:**

| | Fase 1 — classificador | Fase 2 — Fine Tuning |
|:--|:--|:--|
| Camadas treinadas | só o classificador novo (base inteira congelada) | classificador + as **últimas 20 camadas** da base (a partir de `block_15_expand`: blocos 15 e 16 e a convolução final `Conv_1`) |
| Pesos treináveis | ≈1,3 mil | ≈1,2 milhão (53% da base) |
| Taxa de aprendizado | 1e-3 (Adam) | **1e-4** — 10 vezes menor |
| Épocas | até 100, parada por `val_accuracy` (paciência 30) | até 50, parada por `val_accuracy` (paciência 15) |

**Por que congelar até o `block_15_expand`:**

- **As camadas iniciais da MobileNetV2 aprendem características genéricas** — bordas, cores, texturas, contornos simples. Isso serve para qualquer foto, inclusive de tomate e pimentão, então essas camadas ficam congeladas e os pesos da ImageNet são reaproveitados como estão.
- **As camadas finais combinam essas características em padrões específicos das 1.000 classes da ImageNet.** São elas que precisam se adaptar ao nosso domínio (formato, brilho e cor de tomates e pimentões), então são as que descongelamos.
- **Por que só 20 camadas, e não mais:** com apenas 64 fotos de treino, descongelar muitos pesos aumenta o risco de a rede decorar o treino. A partir do `block_15_expand` ficam treináveis cerca de 1,2 milhão de pesos (os blocos 15 e 16 e a `Conv_1`, que trabalham sobre o mapa final de 4×4); descongelar desde o `block_14_expand` (29 camadas) levaria a 1,5 milhão, e desde o `block_13_expand` (38 camadas), a 1,7 milhão.
- **Taxa de aprendizado 10 vezes menor na fase 2:** os pesos da ImageNet já são bons; a ideia é ajustá-los aos poucos, sem apagar o que a rede já sabe.
- **As camadas de `BatchNormalization` continuam em modo de inferência** (a base é chamada com `training=False`): com lotes de só 8 fotos, recalcular as médias dessas camadas desorganizaria o que foi aprendido na ImageNet.
- **Salvaguarda:** se o fine-tuning não alcançar a acurácia de validação da fase 1, o modelo volta para os pesos da fase 1 (o resumo informa qual dos dois foi usado).

In [ ]:
# 5.0 Funções de Transfer Learning + Fine Tuning (usadas nas seções 5 e 6; a avaliação no teste está na 4.0)
CAMADA_INICIO_FINE_TUNING = 'block_15_expand'   # descongela os blocos 15 e 16 e a Conv_1 (20 camadas) — justificativa acima
LR_FASE1 = 1e-3                                 # fase 1: só o classificador novo
LR_FASE2 = LR_FASE1 / 10                        # fase 2 (fine-tuning): 10x menor, para não apagar o que veio da ImageNet
EPOCAS_FASE1, PACIENCIA_FASE1 = 100, 30         # mesmos critérios da CNN da Entrega 2
EPOCAS_FASE2, PACIENCIA_FASE2 = 50, 15


def montar_mobilenet():
    """MobileNetV2 (ImageNet) sem o topo + aumento de dados + classificador novo (probabilidade de ser pimentão)."""
    keras.utils.set_random_seed(0)
    base = keras.applications.MobileNetV2(input_shape=(TAMANHO, TAMANHO, 3), include_top=False, weights='imagenet')
    base.trainable = False
    entrada = keras.Input(shape=(TAMANHO, TAMANHO, 3))
    x = layers.RandomFlip('horizontal_and_vertical')(entrada)   # aumento de dados — ativo só durante o treino
    x = layers.RandomRotation(0.15)(x)
    x = layers.RandomZoom(0.15)(x)
    x = layers.Rescaling(1 / 127.5, offset=-1)(x)              # 0–255 → −1 a 1 (o mesmo que mobilenet_v2.preprocess_input)
    # training=False: as BatchNormalization da base ficam sempre em modo de inferência, também no fine-tuning
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    saida = layers.Dense(1, activation='sigmoid')(x)
    return keras.Model(entrada, saida, name='mobilenetv2_transfer'), base


def treinar_transfer(X, rotulo):
    """Fase 1: base congelada, treina só o classificador. Fase 2: fine-tuning a partir de CAMADA_INICIO_FINE_TUNING."""
    modelo, base = montar_mobilenet()
    dados = dict(x=X['treino'], y=rotulos['treino'], validation_data=(X['validacao'], rotulos['validacao']),
                 batch_size=8, verbose=2)

    # ---- Fase 1: só o classificador novo
    modelo.compile(optimizer=keras.optimizers.Adam(LR_FASE1), loss='binary_crossentropy', metrics=['accuracy'])
    treinaveis_fase1 = sum(int(np.prod(w.shape)) for w in modelo.trainable_weights)
    parada = keras.callbacks.EarlyStopping(monitor='val_accuracy', mode='max', patience=PACIENCIA_FASE1,
                                           restore_best_weights=True)
    inicio = time.perf_counter()
    h1 = modelo.fit(epochs=EPOCAS_FASE1, callbacks=[parada], **dados)
    tempo_fase1 = time.perf_counter() - inicio
    melhor_fase1 = max(h1.history['val_accuracy'])
    pesos_fase1 = modelo.get_weights()

    # ---- Fase 2: fine-tuning — descongela da CAMADA_INICIO_FINE_TUNING até o fim da base
    base.trainable = True
    corte = [camada.name for camada in base.layers].index(CAMADA_INICIO_FINE_TUNING)
    for camada in base.layers[:corte]:
        camada.trainable = False
    # Recompilar é obrigatório depois de mudar o que é treinável (e é aqui que entra a taxa menor)
    modelo.compile(optimizer=keras.optimizers.Adam(LR_FASE2), loss='binary_crossentropy', metrics=['accuracy'])
    treinaveis_fase2 = sum(int(np.prod(w.shape)) for w in modelo.trainable_weights)
    parada = keras.callbacks.EarlyStopping(monitor='val_accuracy', mode='max', patience=PACIENCIA_FASE2,
                                           restore_best_weights=True)
    inicio = time.perf_counter()
    h2 = modelo.fit(epochs=EPOCAS_FASE2, callbacks=[parada], **dados)
    tempo_fase2 = time.perf_counter() - inicio
    melhor_fase2 = max(h2.history['val_accuracy'])

    # Salvaguarda: se o fine-tuning não alcançar a validação da fase 1, volta para os pesos da fase 1
    usou_fine_tuning = melhor_fase2 >= melhor_fase1
    if not usou_fine_tuning:
        modelo.set_weights(pesos_fase1)

    resumo = {
        'épocas fase 1': len(h1.history['loss']), 'melhor val_acc fase 1': round(melhor_fase1, 3),
        'épocas fase 2': len(h2.history['loss']), 'melhor val_acc fase 2': round(melhor_fase2, 3),
        'pesos treináveis fase 1': treinaveis_fase1, 'pesos treináveis fase 2': treinaveis_fase2,
        'tempo de treino (s)': round(tempo_fase1 + tempo_fase2, 1),
        'modelo final': 'com fine-tuning' if usou_fine_tuning else 'fase 1 (fine-tuning não melhorou a validação)',
    }
    print(pd.Series(resumo).to_string())

    # Guardar no Drive (para recarregar sem treinar de novo: keras.models.load_model(...))
    modelo.save(f'{PASTA_IA2}/mobilenetv2_{rotulo}.keras')
    pd.concat([pd.DataFrame(h1.history).assign(fase=1), pd.DataFrame(h2.history).assign(fase=2)]).to_csv(
        f'{PASTA_IA2}/mobilenetv2_{rotulo}_historico.csv', index_label='epoca_na_fase')
    return modelo, resumo


In [ ]:
# 5.1 Treino sem segmentação (fotos inteiras, as mesmas que a CNN da Entrega 2 recebeu)
modelo_tl_sem, resumo_tl_sem = treinar_transfer(X_sem, 'sem_segmentacao')

In [ ]:
# 5.2 Resultado no teste — sem segmentação
tabela_tl_sem, inferencia_tl_sem_ms = avaliar_teste(modelo_tl_sem, X_sem['teste'], preparo=redimensionar)
display(tabela_tl_sem)
print(f"Acurácia no teste: {tabela_tl_sem['acertou?'].sum()}/8 | inferência: {inferencia_tl_sem_ms:.1f} ms por imagem (mediana)")

## 6. Transfer Learning + Fine Tuning com MobileNetV2 — **com** segmentação

Exatamente o modelo, as duas fases e os critérios da seção 5 (mesma camada de corte, mesmas taxas de aprendizado) — só a entrada muda para `X_com`, as imagens recortadas pela U2-Net. Assim, a única diferença entre as seções 5 e 6 é a segmentação.

Na medida de tempo de inferência, a U2-Net entra na conta: cada imagem é segmentada, redimensionada e só então classificada.

In [ ]:
# 6.1 Treino com segmentação (imagens recortadas pela U2-Net na seção 3.2)
modelo_tl_com, resumo_tl_com = treinar_transfer(X_com, 'com_segmentacao')

In [ ]:
# 6.2 Resultado no teste — com segmentação (o tempo inclui a U2-Net)
tabela_tl_com, inferencia_tl_com_ms = avaliar_teste(
    modelo_tl_com, X_com['teste'], preparo=lambda imagem: redimensionar(segmentar_u2net(imagem)[0]))
display(tabela_tl_com)
print(f"Acurácia no teste: {tabela_tl_com['acertou?'].sum()}/8 | inferência: {inferencia_tl_com_ms:.1f} ms por imagem (mediana, com a U2-Net)")

## 7. Tabela comparativa final

As 4 combinações, medidas nas **mesmas 8 imagens de teste**:

| Abordagem | Segmentação | Acurácia no teste | Tempo de treino | Tempo de inferência (por imagem) |
|:--|:--:|:--:|:--:|:--:|
| CNN do zero | não | da Entrega 2 | da Entrega 2 | da Entrega 2 |
| CNN do zero | U2-Net | seção 4 | seção 4 | seção 4 (inclui a U2-Net) |
| MobileNetV2 (Transfer Learning + Fine Tuning) | não | seção 5 | seção 5 | seção 5 |
| MobileNetV2 (Transfer Learning + Fine Tuning) | U2-Net | seção 6 | seção 6 | seção 6 (inclui a U2-Net) |

**Como cada coluna será medida:**

- **acurácia:** acertos nas 8 imagens de teste, com limiar de 0,5 sobre a probabilidade de "pimentão" (o mesmo da Entrega 2);
- **tempo de treino:** só o `fit` (no Transfer Learning, a soma das duas fases) (que inclui a validação a cada época), em GPU T4;
- **tempo de inferência:** mediana por imagem, uma imagem por vez — nas versões com segmentação, o tempo da U2-Net entra na conta, porque ele faz parte do caminho até a resposta;
- a linha da Entrega 2 vem de outra sessão do Colab: os números de tempo dela servem de referência, não de comparação exata.

*(Tabela: a preencher na execução com GPU.)*

## 8. Conclusões

A ser escrita com os números reais da execução, respondendo às duas hipóteses:

1. **A rede pré-treinada (MobileNetV2, com fine-tuning) classificou melhor que a CNN do zero?** — comparando as linhas de mesma segmentação.
2. **A pré-segmentação com a U2-Net ajudou?** — comparando as colunas de mesma rede, e olhando imagem a imagem as respostas de cada combinação.

Com só 8 imagens de teste, cada acerto vale 12,5 pontos percentuais — diferenças pequenas entre combinações não são conclusivas.